# Digital Health Seminar — Choice A: Data Exploration

**Student:** Sarunchana Yongchaiwathana (Ping)  
**Deadline:** 31 May 2026  
**Method:** IVDA methodology — Data → Analysis Tool → Pattern → Context → Knowledge


## 0. Setup & Imports

In [ ]:
import glob
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from scipy import stats

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

DATA_DIR = "../Analysis TAsk"
EXT_DIR  = "../data/external"
FIG_DIR  = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Load Clinical Markers

In [ ]:
clinical = pd.read_csv(f"{DATA_DIR}/ClinicalMarkers_final.csv")

# Clean column names (strip whitespace, lowercase)
clinical.columns = clinical.columns.str.strip().str.lower().str.replace(".", "_", regex=False)

# Derive actual age from birth year
clinical["age"] = 2021 - clinical["age"]

clinical.head()

In [ ]:
clinical["disease_type"].value_counts()

## 2. Load Hourly Sensor Data

In [ ]:
files = glob.glob(f"{DATA_DIR}/Hourly Sensor Data/RHourly_*.csv")
print(f"Found {len(files)} files")

chunks = []
for fp in files:
    pid = int(os.path.basename(fp).replace("RHourly_", "").replace(".csv", ""))
    df = pd.read_csv(fp)
    df["id"] = pid
    chunks.append(df)

sensor = pd.concat(chunks, ignore_index=True)
print(f"Total rows: {len(sensor):,}")

In [ ]:
# Cast heartrate string → float (empty string → NaN)
sensor["heartrate"] = pd.to_numeric(sensor["heartrate"], errors="coerce")

# Parse datetime
sensor["time"] = pd.to_datetime(sensor["time"])
sensor["date"] = sensor["time"].dt.date
sensor["hour"] = sensor["time"].dt.hour
sensor["week"] = sensor["time"].dt.isocalendar().week.astype(int)

sensor.dtypes

## 3. Merge Sensor + Clinical

In [ ]:
df = sensor.merge(clinical[["id", "age", "sex", "disease_type"]], on="id", how="left")
print(df.shape)
df.head()

## 4. Data Quality Check

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print()
print("Heartrate range:", df["heartrate"].min(), "–", df["heartrate"].max())
print("Steps range:", df["steps"].min(), "–", df["steps"].max())
print("Sleep range:", df["sleep"].min(), "–", df["sleep"].max())

In [ ]:
# Rows per participant — check for early dropouts
rows_per_id = df.groupby("id").size().sort_values()
rows_per_id.plot(kind="bar", figsize=(14, 4), title="Hours of data per participant")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/data_coverage.png", dpi=150)
plt.show()

## 5. External Data — Zurich Weather

Source: MeteoSwiss open data (https://opendata.swiss/en/dataset/klimamessnetz-tageswerte)  
Expected columns after loading: `date`, `temp_max`, `temp_mean`, `precipitation`, `sunshine_h`

In [ ]:
# TODO: download and save to data/external/zurich_weather_2021.csv
# weather = pd.read_csv(f"{EXT_DIR}/zurich_weather_2021.csv", parse_dates=["date"])
# df = df.merge(weather, on="date", how="left")
pass

## 6. External Data — Swiss Holidays 2021

Source: Swiss federal calendar / cantonal list for Zurich  
Expected columns: `date`, `is_holiday` (bool), `is_weekend` (bool)

In [ ]:
# TODO: build or download holiday table
# holidays = pd.read_csv(f"{EXT_DIR}/swiss_holidays_2021.csv", parse_dates=["date"])
# df = df.merge(holidays, on="date", how="left")
pass

## 7. External Data — COVID-19 Stringency Index (Switzerland)

Source: Our World in Data — Oxford COVID-19 Government Response Tracker  
Expected columns: `date`, `stringency_index`

In [ ]:
# TODO: filter OWID data for Switzerland, Apr–Sep 2021
# covid = pd.read_csv(f"{EXT_DIR}/covid_stringency_ch_2021.csv", parse_dates=["date"])
# df = df.merge(covid, on="date", how="left")
pass

---
## Finding 1 — Activity Patterns by Time of Day × Disease Stage

**Question:** Do patients with different disease stages show different daily activity rhythms?  
**Visualization:** Heatmap of average hourly steps — rows = disease stage, columns = hour of day

In [ ]:
heatmap_data = (
    df.groupby(["disease_type", "hour"])["steps"]
    .mean()
    .unstack("hour")
)

fig, ax = plt.subplots(figsize=(14, 4))
sns.heatmap(heatmap_data, cmap="YlOrRd", ax=ax, linewidths=0.3)
ax.set_title("Finding 1: Mean hourly steps by hour-of-day and disease stage")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Disease stage")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/finding1_activity_heatmap.png", dpi=150)
plt.show()

---
## Finding 2 — Weather Effect on Physical Activity

**Question:** Do warmer, sunnier days correlate with higher step counts?  
**Visualization:** Scatter of daily steps vs. max temperature, colored by disease stage

> Requires weather data to be loaded in Section 5.

In [ ]:
# Aggregate to daily steps per participant
daily = (
    df.groupby(["id", "date", "disease_type"])
    .agg(daily_steps=("steps", "sum"), daily_hr=("heartrate", "mean"), daily_sleep=("sleep", "sum"))
    .reset_index()
)

# TODO: after merging weather, uncomment:
# fig = px.scatter(daily, x="temp_max", y="daily_steps", color="disease_type",
#                  trendline="lowess", opacity=0.5,
#                  title="Finding 2: Daily steps vs. max temperature")
# fig.show()
daily.head()

---
## Finding 3 — Sleep & Heart Rate Trends Over Study Period

**Question:** How do sleep minutes and resting HR evolve week-by-week?  
**Visualization:** Smoothed weekly time series of group-average sleep and HR

In [ ]:
weekly = (
    df.groupby(["week", "disease_type"])
    .agg(mean_sleep=("sleep", "mean"), mean_hr=("heartrate", "mean"))
    .reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for stage, grp in weekly.groupby("disease_type"):
    axes[0].plot(grp["week"], grp["mean_sleep"], marker="o", label=stage)
    axes[1].plot(grp["week"], grp["mean_hr"],    marker="o", label=stage)

axes[0].set_title("Mean hourly sleep minutes by week")
axes[0].set_xlabel("ISO week")
axes[0].legend()

axes[1].set_title("Mean resting heart rate by week")
axes[1].set_xlabel("ISO week")
axes[1].legend()

plt.suptitle("Finding 3: Sleep & HR trends over study period")
plt.tight_layout()
plt.savefig(f"{FIG_DIR}/finding3_sleep_hr_trends.png", dpi=150)
plt.show()

---
## Optional Finding 4 — COVID Stringency & Step Counts

> Requires COVID stringency data to be loaded in Section 7.

In [ ]:
# TODO: after merging covid stringency, uncomment:
# daily_all = daily.groupby("date").agg(mean_steps=("daily_steps", "mean")).reset_index()
# daily_all = daily_all.merge(covid, on="date", how="left")
# fig = go.Figure()
# fig.add_trace(go.Scatter(x=daily_all["date"], y=daily_all["mean_steps"], name="Mean daily steps"))
# fig.add_trace(go.Scatter(x=daily_all["date"], y=daily_all["stringency_index"],
#                          name="COVID stringency", yaxis="y2"))
# fig.update_layout(title="Finding 4: Steps vs. COVID stringency",
#                   yaxis2=dict(overlaying="y", side="right"))
# fig.show()
pass

---
## Bonus — Clustering by Activity Profile (k-means + PCA)

Earns +0.25 grade bonus if submitted.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Build per-participant feature vector
features = (
    daily.groupby("id")
    .agg(mean_steps=("daily_steps", "mean"), std_steps=("daily_steps", "std"),
         mean_hr=("daily_hr", "mean"), mean_sleep=("daily_sleep", "mean"))
    .dropna()
    .reset_index()
)

X = StandardScaler().fit_transform(features[["mean_steps", "std_steps", "mean_hr", "mean_sleep"]])

# TODO: choose k via elbow/silhouette, then fit:
# kmeans = KMeans(n_clusters=3, random_state=42).fit(X)
# features["cluster"] = kmeans.labels_

pca = PCA(n_components=2)
coords = pca.fit_transform(X)
features["pc1"] = coords[:, 0]
features["pc2"] = coords[:, 1]

# Merge disease stage for coloring
features = features.merge(clinical[["id", "disease_type"]], on="id", how="left")

fig = px.scatter(features, x="pc1", y="pc2", color="disease_type",
                 title="Bonus: PCA of participant activity profiles (colored by disease stage)",
                 labels={"pc1": f"PC1 ({pca.explained_variance_ratio_[0]:.1%})",
                         "pc2": f"PC2 ({pca.explained_variance_ratio_[1]:.1%})"})
fig.show()